# Extracción y Curaduría de Datos de Presencia desde la API de GBIF
**Banco de Germoplasma de Especies Nativas (BGEN-Noreste) — UNAJ**

Este cuaderno de Jupyter documenta el proceso de extracción, limpieza y consolidación de datos de ocurrencia de 19 especies nativas de interés para la conservación en la región Pampeana y bonaerense, utilizando la API de la Infraestructura Mundial de Información en Biodiversidad (GBIF).

In [ ]:
# Instalación de librerías requeridas en entornos interactivos (ej. Google Colab)
!pip install pandas geopandas requests tqdm shapely

In [ ]:
import os
import time
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
from urllib.parse import quote

print("Librerías importadas con éxito.")

## 1. Función para Consulta y Descarga desde la API de GBIF
Definimos una función que realiza peticiones paginadas a la API de GBIF para descargar los registros de presencia de una especie en un país determinado (en este caso, Argentina: "AR").

In [ ]:
def get_gbif_species_data(species_name, country_code="AR"):
    """
    Consulta la API de GBIF y descarga registros de presencia para una especie en un país dado.
    
    Parameters:
        species_name (str): Nombre científico de la especie.
        country_code (str): Código ISO del país (por defecto "AR" para Argentina).
        
    Returns:
        pd.DataFrame: DataFrame de pandas con los registros obtenidos.
    """
    base_url = "https://api.gbif.org/v1/occurrence/search"
    limit = 300
    offset = 0
    all_records = []
    
    headers = {"User-Agent": "BGEN-UNAJ-Scholarship/1.0"}
    
    print(f"Iniciando descarga para: {species_name}...")
    
    while True:
        params = {
            "scientificName": species_name,
            "country": country_code,
            "limit": limit,
            "offset": offset,
            "hasCoordinate": "true"  # Filtrar solo registros con coordenadas espaciales
        }
        
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=15)
            if response.status_code != 200:
                print(f"Error de servidor en offset {offset}: código {response.status_code}")
                break
                
            data = response.json()
            results = data.get("results", [])
            
            if not results:
                break
                
            for rec in results:
                all_records.append({
                    "species": rec.get("species", species_name),
                    "decimalLatitude": rec.get("decimalLatitude"),
                    "decimalLongitude": rec.get("decimalLongitude"),
                    "eventDate": rec.get("eventDate"),
                    "datasetName": rec.get("datasetName", "GBIF"),
                    "catalogNumber": rec.get("catalogNumber")
                })
            
            if data.get("endOfRecords", True):
                break
                
            offset += limit
            time.sleep(0.5)  # Respetar tasa de peticiones del servidor
            
        except Exception as e:
            print(f"Excepción en offset {offset}: {e}")
            break
            
    df = pd.DataFrame(all_records)
    print(f"Descarga finalizada. Registros obtenidos: {len(df)}")
    return df

## 2. Ejecución de Consultas para las 19 Especies del BGEN
Definimos la lista de las 19 especies vegetales nativas bajo estudio y descargamos sus datos de ocurrencia de forma iterativa.

In [ ]:
species_list = [
    "Araujia sericifera",
    "Austroeupatorium inulifolium",
    "Ceiba speciosa",
    "Celtis tala",
    "Cortaderia selloana",
    "Duranta erecta",
    "Erythrina crista-galli",
    "Ipomoea alba",
    "Jacaranda mimosifolia",
    "Passiflora caerulea",
    "Phytolacca dioica",
    "Salpichroa origanifolia",
    "Schinus molle",
    "Senna corymbosa",
    "Solanum pseudocapsicum",
    "Syagrus romanzoffiana",
    "Tecoma stans",
    "Tipuana tipu",
    "Vachellia caven"
]

# Contenedor para consolidar los resultados
dfs = []

for sp in tqdm(species_list, desc="Procesando especies"):
    df_sp = get_gbif_species_data(sp, country_code="AR")
    if not df_sp.empty:
        dfs.append(df_sp)
        
if dfs:
    gbif_df = pd.concat(dfs, ignore_index=True)
    print(f"Consolidado de GBIF finalizado. Total registros brutos con coordenadas: {len(gbif_df)}")
else:
    gbif_df = pd.DataFrame()
    print("No se obtuvieron registros de ninguna especie.")

## 3. Limpieza de Datos y Guardado de la Base de Ocurrencias
Realizamos la normalización final de la base de datos, eliminamos registros duplicados y guardamos los resultados en un archivo CSV listo para su importación en R (pipeline biomod2).

In [ ]:
if not gbif_df.empty:
    # 1. Eliminar coordenadas nulas o vacías
    clean_df = gbif_df.dropna(subset=["decimalLatitude", "decimalLongitude"])
    
    # 2. Filtrar coordenadas absurdas o fuera de límites lógicos de Argentina
    clean_df = clean_df[(clean_df["decimalLatitude"] >= -56.0) & (clean_df["decimalLatitude"] <= -21.0)]
    clean_df = clean_df[(clean_df["decimalLongitude"] >= -74.0) & (clean_df["decimalLongitude"] <= -53.0)]
    
    # 3. Eliminar duplicados espaciales idénticos
    clean_df = clean_df.drop_duplicates(subset=["species", "decimalLatitude", "decimalLongitude"])
    
    # 4. Guardar base de datos consolidada
    output_filename = "Especies_Nativas_AR.csv"
    clean_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
    print(f"Base curada guardada exitosamente en '{output_filename}'. Total registros limpios: {len(clean_df)}")
else:
    print("No hay datos para procesar.")

## 4. Conversión y Exportación Espacial (Geopandas)
Convertimos los registros a puntos espaciales y los exportamos en formato GeoJSON para su integración con QGIS y QField.

In [ ]:
if os.path.exists("Especies_Nativas_AR.csv"):
    df_spatial = pd.read_csv("Especies_Nativas_AR.csv")
    
    # Crear geometría de puntos
    geometry = [Point(xy) for xy in zip(df_spatial["decimalLongitude"], df_spatial["decimalLatitude"])]
    
    # Crear GeoDataFrame con CRS WGS84 (EPSG:4326)
    gdf = gpd.GeoDataFrame(df_spatial, geometry=geometry, crs="EPSG:4326")
    
    # Guardar en formato GeoJSON
    geojson_filename = "Especies_Nativas_AR.geojson"
    gdf.to_file(geojson_filename, driver="GeoJSON")
    print(f"Capa vectorial guardada exitosamente en '{geojson_filename}'")
else:
    print("El archivo CSV de origen no existe.")